# Question B：使用 Funnelling Approach 进行特征选择

本题在沪深 300 指数数据上构造滚动收益和滚动波动率特征，并将 filter、wrapper 和 embedded 三类方法组合成漏斗式特征选择流程。

In [ ]:
# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt

# Classifier
from xgboost import XGBClassifier

# Feature selection and model validation
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import train_test_split, TimeSeriesSplit, cross_val_score
from sklearn.utils.class_weight import compute_sample_weight

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Arial Unicode MS", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

## 数据读取与候选特征构造

候选特征与建模流程保持一致：先计算日度对数收益率，然后生成 10 到 60 日窗口的滚动累计收益 `Ret_*` 和滚动标准差 `Std_*`。这些特征分别刻画趋势/动量和波动状态。

In [ ]:
# Load file
df = pd.read_csv("CSI300_2005_2026.csv", index_col=0, parse_dates=True)

# Calculate returns
df["Returns"] = np.log(df["Adj Close"]).diff()
df = df["2010":].copy()

# Create features (predictors) list
features_list = []
for r in range(10, 65, 5):
    df["Ret_" + str(r)] = df.Returns.rolling(r).sum()
    df["Std_" + str(r)] = df.Returns.rolling(r).std()
    features_list.append("Ret_" + str(r))
    features_list.append("Std_" + str(r))

# Define target before dropping NaN values so the last row without a forward return is removed.
target_threshold = 0.0010
df["Target_Return"] = np.log(df["Adj Close"].shift(-1) / df["Adj Close"])
df["Label"] = np.where(df["Target_Return"] > target_threshold, 1, 0)

# Drop NaN values
df.dropna(inplace=True)

X = df[features_list]
y = df["Label"].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)
sample_weights = compute_sample_weight(class_weight="balanced", y=y_train)

summary = pd.DataFrame({
    "Item": ["Candidate features", "Training observations", "Testing observations", "Training positive rate", "Testing positive rate"],
    "Value": [len(features_list), len(X_train), len(X_test), round(y_train.mean(), 4), round(y_test.mean(), 4)],
})
summary

,Item,Value
0,Candidate features,22.0000
1,Training observations,3130.0000
2,Testing observations,783.0000
3,Training positive rate,0.4684
4,Testing positive rate,0.4393


## Step 1：Filter 方法

Filter 阶段先删除高度相关的冗余特征，再用 mutual information 对剩余特征排序。相关性阈值设为 0.98，用于减少几乎重复的滚动波动率特征。

In [ ]:
# Correlation filter
corr = X_train.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
corr_dropped = [column for column in upper.columns if any(upper[column] > 0.98)]
corr_features = [column for column in X_train.columns if column not in corr_dropped]

# Mutual information filter
mi_scores = pd.Series(
    mutual_info_classif(X_train[corr_features], y_train, random_state=42),
    index=corr_features,
).sort_values(ascending=False)

filter_features = list(mi_scores.head(min(18, len(mi_scores))).index)

print(f"Initial feature count: {len(features_list)}")
print(f"Dropped by correlation filter: {corr_dropped}")
print(f"Features kept for wrapper step: {len(filter_features)}")
mi_scores.head(18).to_frame("mutual_information")

Initial feature count: 22
Dropped by correlation filter: ['Std_35', 'Std_40', 'Std_45', 'Std_50', 'Std_55', 'Std_60']
Features kept for wrapper step: 16


,mutual_information
Ret_60,0.020599
Ret_40,0.018756
Ret_55,0.011541
Ret_10,0.011207
Ret_20,0.008496
Ret_50,0.005738
Std_30,0.000944
Ret_30,0.000811
Std_25,0.000000
Ret_25,0.000000


## Step 2：Wrapper 方法

Wrapper 阶段使用 XGBoost 分类器和 `TimeSeriesSplit`。按互信息排序取前 N 个特征，比较不同 N 下的交叉验证 ROC AUC，并使用 `compute_sample_weight` 处理训练集类别不平衡。

In [ ]:
# Wrapper selection with time-series cross validation
tscv = TimeSeriesSplit(n_splits=5, gap=1)
wrapper_rows = []

for n_features in [8, 10, 12, 14, 16, 18]:
    cols = filter_features[: min(n_features, len(filter_features))]
    selector_model = XGBClassifier(
        verbosity=0,
        eval_metric="logloss",
        n_estimators=100,
        max_depth=3,
        learning_rate=0.05,
    )
    cv_scores = cross_val_score(
        selector_model,
        X_train[cols],
        y_train,
        cv=tscv,
        scoring="roc_auc",
        params={"sample_weight": sample_weights},
        n_jobs=1,
    )
    wrapper_rows.append({
        "n_features": len(cols),
        "cv_roc_auc_mean": cv_scores.mean(),
        "cv_roc_auc_std": cv_scores.std(),
        "features": cols,
    })

wrapper_table = pd.DataFrame(wrapper_rows).drop_duplicates("n_features")
best_wrapper = wrapper_table.sort_values(["cv_roc_auc_mean", "n_features"], ascending=[False, True]).iloc[0]
wrapper_features = list(best_wrapper["features"])

wrapper_table.drop(columns=["features"]).round(4)

,n_features,cv_roc_auc_mean,cv_roc_auc_std
0,8,0.5130,0.0131
1,10,0.5073,0.0194
2,12,0.5032,0.0194
3,14,0.5114,0.0179
4,16,0.5007,0.0130


## Step 3：Embedded 方法

Embedded 阶段在 wrapper 选出的特征上训练 XGBoost，并使用模型内部的 gain importance 排序。最终保留 gain 排名前 10 的特征作为第 3 题模型输入。

In [ ]:
# Embedded selection by XGBoost gain importance
embedded_model = XGBClassifier(
    verbosity=0,
    eval_metric="logloss",
    n_estimators=100,
    max_depth=3,
    learning_rate=0.05,
    importance_type="gain",
)

embedded_model.fit(X_train[wrapper_features], y_train, sample_weight=sample_weights)

gain_scores = pd.Series(
    embedded_model.feature_importances_,
    index=wrapper_features,
).sort_values(ascending=False)

final_features = list(gain_scores.head(min(10, len(gain_scores))).index)

final_feature_table = pd.DataFrame({
    "Rank": range(1, len(final_features) + 1),
    "Feature": final_features,
    "XGBoost gain": gain_scores.loc[final_features].round(6).values,
})
final_feature_table

,Rank,Feature,XGBoost gain
0,1,Ret_30,0.167559
1,2,Ret_10,0.132976
2,3,Ret_40,0.131482
3,4,Ret_55,0.125410
4,5,Ret_60,0.124697
5,6,Ret_50,0.115577
6,7,Std_30,0.101566
7,8,Ret_20,0.100733


## 特征选择结论

漏斗流程从 22 个候选滚动特征开始，先通过相关性和互信息进行 filter，再通过时间序列交叉验证完成 wrapper 筛选，最后通过 XGBoost gain importance 完成 embedded 筛选。最终特征将用于第 3 题的梯度提升模型。